In [2]:
from load_results import load_result_dataset
import pandas as pd

pn1 = 'full_fine_tuning_50epochs_edge_paper_final2'
pn2 = 'full_fine_tuning_50epochs_paper_final2'
pn3 = 'none'
final_data1 = load_result_dataset(pn1, pn2, pn3)
final_data1 = [{**d, 'ft_strategy': 'FFT (50 epochs)'} for d in final_data1]
df1 = pd.DataFrame(final_data1)

pn1 = 'full_fine_tuning_5epochs_edge_article1'
pn2 = 'full_fine_tuning_5epochs_article1'
pn3 = 'none'
final_data2 = load_result_dataset(pn1, pn2, pn3)
final_data2 = [{**d, 'ft_strategy': 'FFT (5 epochs)'} for d in final_data2]
df2 = pd.DataFrame(final_data2)

pn1 = 'linearprobe_50epochs_edge_paper_final2'
pn2 = 'linearprobe_50epochs_paper_final2'
pn3 = 'none'
final_data3 = load_result_dataset(pn1, pn2, pn3)
final_data3 = [{**d, 'ft_strategy': 'LP (50 epochs)'} for d in final_data3]
df3 = pd.DataFrame(final_data3)

df = pd.concat([ df1, df2, df3 ], axis=0, ignore_index=True) 

final_data = []
final_data.extend(final_data1)
final_data.extend(final_data2)
final_data.extend(final_data3)

../results/none/CLIP-convnext_base_w-laion_aesthetic-s13B-b82K_uc-merced-land-use-dataset_TRADES_v2.pkl
HEY
../results/full_fine_tuning_50epochs_paper_final2/CLIP-convnext_base_w-laion_aesthetic-s13B-b82K_uc-merced-land-use-dataset_TRADES_v2.pkl
../results/none/CLIP-convnext_base_w-laion2B-s13B-b82K_uc-merced-land-use-dataset_TRADES_v2.pkl
HEY
../results/full_fine_tuning_50epochs_paper_final2/CLIP-convnext_base_w-laion2B-s13B-b82K_uc-merced-land-use-dataset_TRADES_v2.pkl
../results/none/deit_small_patch16_224.fb_in1k_uc-merced-land-use-dataset_TRADES_v2.pkl
HEY
../results/full_fine_tuning_50epochs_paper_final2/deit_small_patch16_224.fb_in1k_uc-merced-land-use-dataset_TRADES_v2.pkl
../results/none/robust_resnet50_uc-merced-land-use-dataset_TRADES_v2.pkl
HEY
../results/full_fine_tuning_50epochs_paper_final2/robust_resnet50_uc-merced-land-use-dataset_TRADES_v2.pkl
../results/none/vit_small_patch16_224.augreg_in21k_uc-merced-land-use-dataset_TRADES_v2.pkl
HEY
../results/full_fine_tuning_50

In [2]:

from process_database import process_grouped_df, process_rankings
import pandas as pd
import numpy as np
from pathlib import Path
pd.set_option("display.max_colwidth", None)   # no truncation for any column

# ── helper: make a two‑line cell ────────────────────────────────────────────
def fmt_entry(backbone: str, loss: str, rank: int) -> str:
    backbone = backbone.replace('_', r'\_')
    loss     = loss.replace('_', r'\_')
    return (
        r'\makecell{'
        f'{backbone}, {loss} \\\\[0.3ex] '     # ← four “\” in Python ⇒ “\\” in .tex
        r'\footnotesize (GR:' f'{rank}' r')'
        r'}'
    )

from pathlib import Path
import pandas as pd
import numpy as np

final_data = pd.concat([df1, df2, df3], ignore_index=True)

for setting in ("FFT (50 epochs)", "FFT (5 epochs)", "LP (50 epochs)" ): 
    rows = []
    d = final_data[final_data.ft_strategy == setting]
    g = process_rankings( process_grouped_df(d) )

    for size in ( "Tiny","Small","Base", ): # 
        copy = g.xs(size, level="model_size")

        # ── flatten & rename columns ────────────────────────────────────────
        cols_keep = ["rank_borda", "score_sum", ] #"rank_geom", "rank_sum", "score_geom",  "borda"
        df_bis = (copy.loc[:, ("TOTAL", cols_keep)].reset_index())

        df_bis.columns = ["_".join(c) if isinstance(c, tuple) else c for c in df_bis.columns]
        df_bis.columns = ["backbone", "backbone_name", "loss",
                      "pre_training_strategy", "model_type", "ft_strategy",
                      "rank_borda_tot",  "score_sum_tot", ] #"rank_geom_tot", "rank_sum_tot", "score_geom_tot", "score_borda_tot"

        # ── choose top‑3 by total‑sum score ────────────────────────────────
        df_bis = (df_bis.round(4)
                .sort_values("rank_borda_tot", ascending=True)
                .head(3)
                .reset_index(drop=True))

        rows.append({
            "Size": size,
            "Gold (1st)"  : fmt_entry(df_bis.loc[0, "backbone_name"],
                                      df_bis.loc[0, "loss"],
                                      int(df_bis.loc[0, "rank_borda_tot"]) ),
            "Silver (2nd)": fmt_entry(df_bis.loc[1, "backbone_name"],
                                      df_bis.loc[1, "loss"],
                                      int(df_bis.loc[1, "rank_borda_tot"]) ),
            "Bronze (3rd)": fmt_entry(df_bis.loc[2, "backbone_name"],
                                      df_bis.loc[2, "loss"],
                                      int( df_bis.loc[2, "rank_borda_tot"])  ),
            # r"$\Delta \%$ Abs. perf.  (1st$\rightarrow$2nd)": df_bis.loc[1, "score_sum_tot"] - df_bis.loc[0, "score_sum_tot"],
            # r"$\Delta \%$ Abs. perf. (2nd$\rightarrow$3rd)": df_bis.loc[2, "score_sum_tot"] - df_bis.loc[1, "score_sum_tot"]
        })

    table = pd.DataFrame(rows).round(2)
    table.columns = [rf"\textbf{{{c}}}" for c in table.columns]

    body = table.to_latex(index=False,
                          escape=False,
                          column_format="|" + "|".join(["c"]*table.shape[1]) + "|")

    safe_label = (setting.replace(" ", "_")
                         .replace("(", "")
                         .replace(")", ""))

    # ── only wrap in \resizebox if the table is wide (> 1.0× textwidth) ──
    maybe_resize_open  = r"\resizebox{\textwidth}{!}{%"  # default on
    maybe_resize_close = r"}"

    # crude width check (char count) – feel free to refine
    approx_char_per_col = 18
    if approx_char_per_col * table.shape[1] < 140:   # fits? skip resizebox
        maybe_resize_open = maybe_resize_close = ""

    latex_table = "\n".join([
        r"\begin{table}[ht]",
        r"\centering",
        rf"\caption{{Top fine‑tuning configurations in {setting}, with global ranking (GR) below. }}",
        rf"\label{{tab:{safe_label}}}",
        r"\resizebox{\textwidth}{!}{%",      #  ← open
        body,                                #  ← pandas output
        r"}",                                #  ← close
        r"\end{table}",
        ""
    ])

    out = Path("latex_tables") / f"{safe_label}.tex"
    out.write_text(latex_table, encoding="utf-8")
    print("Saved:", out)


Percentage of NaN values: 0.00%
Saved: latex_tables/FFT_50_epochs.tex
Percentage of NaN values: 0.00%
Saved: latex_tables/FFT_5_epochs.tex
Percentage of NaN values: 0.00%
Saved: latex_tables/LP_50_epochs.tex


/var/folders/v7/3s0lms795672_f7_mh2x6bcr0000gn/T/ipykernel_18336/443611353.py:65: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  body = table.to_latex(index=False,
/var/folders/v7/3s0lms795672_f7_mh2x6bcr0000gn/T/ipykernel_18336/443611353.py:65: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  body = table.to_latex(index=False,
/var/folders/v7/3s0lms795672_f7_mh2x6bcr0000gn/T/ipykernel_18336/443611353.py:65: FutureWarning: In future versions `DataFrame.to_latex` is expected to 

In [4]:
from process_database import process_grouped_df, process_rankings
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_colwidth", None)


# ─────────────────────────────────────────────────────────────────────────────
# LaTeX preamble requirements (add these to your main .tex file once):
#
#   \usepackage{multirow}
#   \usepackage{booktabs}
#   \usepackage{xcolor}
#   \usepackage{tcolorbox}   % for \lossbadge
#
#   \definecolor{scoregray}{gray}{0.45}
#   \definecolor{tradescolor}{HTML}{2E7D32}    % green-ish for TRADES
#   \definecolor{atcolor}{HTML}{1565C0}        % blue-ish for Classic AT
#
#   \newcommand{\lossbadge}[1]{%
#     \tikz[baseline=(b.base)]\node[
#       inner sep=1.5pt, rounded corners=2pt,
#       fill=\ifstrequal{#1}{TRADES}{tradescolor!15}{atcolor!15},
#       text=\ifstrequal{#1}{TRADES}{tradescolor}{atcolor},
#       font=\scriptsize\sffamily\bfseries
#     ] (b) {#1};%
#   }
#
# (or define \lossbadge however you like — the script just calls \lossbadge{TRADES}
# or \lossbadge{Classic AT})
# ─────────────────────────────────────────────────────────────────────────────


def escape_latex(s: str) -> str:
    """Escape underscores for LaTeX — applied only to backbone shortcuts."""
    return s.replace("_", r"\_")


def build_size_block(size_label: str, top3: pd.DataFrame) -> str:
    """
    Build the three text-rows for one size bucket:
      row 1: \texttt{shortcut} (gold | silver | bronze)
      row 2: \lossbadge{loss}
      row 3: GR / BS / SS metadata in scoregray
    """
    # Pull the three medal entries out as plain Python values
    medals = []
    for i in range(3):
        backbone = escape_latex(str(top3.loc[i, "backbone_name"]))
        loss     = str(top3.loc[i, "loss"])
        gr       = int(top3.loc[i, "rank_borda_tot"])
        bs       = int(top3.loc[i, "borda_tot"]) if "borda_tot" in top3.columns else int(top3.loc[i, "score_borda_tot"])
        ss       = float(top3.loc[i, "score_sum_tot"])
        medals.append({"backbone": backbone, "loss": loss, "gr": gr, "bs": bs, "ss": ss})

    # Row 1: shortcut names
    row1 = (
        rf"\multirow{{3}}{{*}}{{{size_label}}}" "\n"
        rf"  & \texttt{{\small {medals[0]['backbone']}}}"  "\n"
        rf"  & \texttt{{\small {medals[1]['backbone']}}}"  "\n"
        rf"  & \texttt{{\small {medals[2]['backbone']}}} \\"
    )

    # Row 2: loss badges
    row2 = (
        rf"  & \lossbadge{{{medals[0]['loss']}}}"  "\n"
        rf"  & \lossbadge{{{medals[1]['loss']}}}"  "\n"
        rf"  & \lossbadge{{{medals[2]['loss']}}} \\"
    )

    # Row 3: GR / BS / SS metadata (scriptsize, scoregray)
    def meta(m):
        return (rf"{{\scriptsize\color{{scoregray}} "
                rf"GR:{m['gr']}\,\textbar\,BS:{m['bs']}\,\textbar\,SS:{m['ss']:.2f}}}")

    row3 = (
        rf"  & {meta(medals[0])}"  "\n"
        rf"  & {meta(medals[1])}"  "\n"
        rf"  & {meta(medals[2])} \\"
    )

    return "\n".join([row1, row2, row3])


# ─────────────────────────────────────────────────────────────────────────────
# Main loop
# ─────────────────────────────────────────────────────────────────────────────
final_data = pd.concat([df1, df2, df3], ignore_index=True)

# Pretty captions per setting
caption_short = {
    "FFT (50 epochs)": "FFT-50",
    "FFT (5 epochs)":  "FFT-5",
    "LP (50 epochs)":  "LP-50",
}

for setting in ("FFT (50 epochs)", "FFT (5 epochs)", "LP (50 epochs)"):
    d = final_data[final_data.ft_strategy == setting]
    g = process_rankings(process_grouped_df(d))

    size_blocks = []
    for size in ("Tiny", "Small", "Base"):
        copy = g.xs(size, level="model_size")

        # Flatten the multi-index columns we care about
        cols_keep = ["rank_borda", "score_sum", "borda"]
        df_bis = copy.loc[:, ("TOTAL", cols_keep)].reset_index()

        df_bis.columns = ["_".join(c) if isinstance(c, tuple) else c
                          for c in df_bis.columns]
        df_bis.columns = [
            "backbone", "backbone_name", "loss",
            "pre_training_strategy", "model_type", "ft_strategy",
            "rank_borda_tot", "score_sum_tot", "borda_tot",
        ]

        top3 = (df_bis.round(4)
                      .sort_values("rank_borda_tot", ascending=True)
                      .head(3)
                      .reset_index(drop=True))

        size_blocks.append(build_size_block(size, top3))

    # Glue size blocks together with a small vertical gap
    # body = "\n\\\\[6pt]\n".join(size_blocks)
    # body = "\n\\addlinespace\n".join(size_blocks)
    body = "\n".join(size_blocks)              # just glue them together
    # Note: the join above adds extra `\\[6pt]` between blocks; the last row of
    # each block already ends with `\\`, so the join produces `\\\n\\[6pt]\n...`
    # which renders as a `\\` line break followed by `[6pt]` extra spacing on the
    # next row. If your LaTeX compiler complains about the empty `\\` before
    # `[6pt]`, switch to a simple "\n\\addlinespace[6pt]\n".join(...).

    safe_label = (setting.replace(" ", "_")
                          .replace("(", "")
                          .replace(")", ""))
    short_name = caption_short[setting]

    caption = (
        rf"\small Top {short_name} configurations per architecture size, "
        rf"ranked by Borda score (BS). Global ranking (GR) and sum score (SS) "
        rf"also reported. Mapping to Hugging Face model cards available in "
        rf"Table \ref{{tab:backbone_config_refs}}."
    )

    latex_table = "\n".join([
        r"\begin{table}[ht]",
        r"\centering",
        r"\footnotesize",
        r"\setlength{\tabcolsep}{3pt}",
        rf"\caption{{{caption}}}",
        r"\begin{tabular}{@{}l ccc@{}}",
        r"\toprule",
        r"\textbf{Size}",
        r"  & \textbf{Gold (1st)}",
        r"  & \textbf{Silver (2nd)}",
        r"  & \textbf{Bronze (3rd)} \\",
        r"\midrule",
        body,
        r"\bottomrule",
        r"\end{tabular}",
        rf"\label{{tab:{safe_label}}}",
        r"\end{table}",
        "",
    ])

    out_dir = Path("latex_tables")
    out_dir.mkdir(exist_ok=True)
    out = out_dir / f"{safe_label}.tex"
    out.write_text(latex_table, encoding="utf-8")
    print("Saved:", out)

Percentage of NaN values: 0.00%
Saved: latex_tables/FFT_50_epochs.tex
Percentage of NaN values: 0.00%
Saved: latex_tables/FFT_5_epochs.tex
Percentage of NaN values: 0.00%
Saved: latex_tables/LP_50_epochs.tex
